# One fine-tuned model for routing and choice questions: LoRA + a scoring head, ModernBERT-large vs Nemotron-1B

**What.** Run 18 (`choice_lora.ipynb`) showed that LoRA fine-tuning lifts general choice questions from 114 to 153 / 170 (E12). Here **one** model does both jobs:
- **agent routing** with chat history and the current agent: "Which agent should handle the user's latest message?";
- **general choice questions** about a text: yes / no, one of N options, or a level on a scale (the decider's job).

The model is one encoder with LoRA adapters plus one scoring head, trained on both kinds of question. Each question is presented in the same format, with one pass per option:

```text
<conversation or text> [SEP] current agent: X [SEP] question: <q> [SEP] candidate: <option>   -> encoder (+ LoRA) -> mean -> head -> score
```

It is run twice in this notebook, with the same data, settings and tests: **ModernBERT-large** (395 M) first, then **Nemotron-1B** (`llama-nemotron-embed-1b-v2`, 1.2 B, bidirectional Llama).

**Tests** (all never used for training): E10 realistic chat history (193 turns, live; the 83 test-company turns are the headline), E11 first turns (100), banking (50 / 225) and E12 choice questions (170). **Latency** is measured for every test: one question at a time, warm, tokenization included, p50 / p95 / mean in milliseconds on the T4.

**Compared with:** the frozen-encoder system (run 17), run 18 (choice-only LoRA) and the Strands decider.

## 1 — Setup

| Setting | Default | Meaning |
|---|---|---|
| `RUN_ENCODERS` | `modernbert-large,nemotron-1b` | encoders to train, in order |
| `LORA_R`, `LORA_ALPHA` | 16, 32 | LoRA rank and scale |
| `EPOCHS` | 2 | passes over all training questions |
| `LR`, `HEAD_LR` | 2e-4, 1e-3 | learning rates of the adapters and of the head |
| `BATCH_ROUTE`, `BATCH_CHOICE` | 4, 16 | questions per step (a routing question brings 5 longer inputs, a choice question 2–5 short ones) |
| `MAX_LEN` | 320 | tokens per input; the longest routing input is about 255. If anything is cut, it is the start of the history, never the candidate |
| `LORA_SMOKE` | 0 | 1 = a tiny code check on a CPU |

In [1]:
!pip install -q "transformers==5.17.0"
import copy, gc, json, math, os, random, statistics, subprocess, time
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

SMOKE = os.environ.get("LORA_SMOKE") == "1"
RUN_ENCODERS = os.environ.get("RUN_ENCODERS", "modernbert-base" if SMOKE else "modernbert-large,nemotron-1b").split(",")
LORA_R, LORA_ALPHA, LORA_DROPOUT = int(os.environ.get("LORA_R", "16")), float(os.environ.get("LORA_ALPHA", "32")), 0.1
EPOCHS = int(os.environ.get("EPOCHS", "1" if SMOKE else "2"))
LR, HEAD_LR = float(os.environ.get("LR", "2e-4")), float(os.environ.get("HEAD_LR", "1e-3"))
BATCH_ROUTE, BATCH_CHOICE = int(os.environ.get("BATCH_ROUTE", "4")), int(os.environ.get("BATCH_CHOICE", "16"))
MAX_LEN = int(os.environ.get("MAX_LEN", "320"))
SEED = int(os.environ.get("SEED", "42"))
K_MAX = 5
ROUTE_Q = "Which agent should handle the user's latest message?"

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
RUN_LOG = Path("lora_combined_run.log")


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")


def gpu():
    return f"GPU {torch.cuda.memory_allocated(DEVICE) / 1e9:.1f} GB (peak {torch.cuda.max_memory_allocated(DEVICE) / 1e9:.1f})" if AMP else "CPU"


import transformers
log(f"torch {torch.__version__} | transformers {transformers.__version__} | device {DEVICE}"
    + (f" ({torch.cuda.get_device_name(DEVICE)})" if AMP else "") + f" | encoders {RUN_ENCODERS}" + (" | SMOKE CHECK" if SMOKE else ""))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 95.3 MB/s eta 0:00:00:00:010:01
[15:52:28] torch 2.11.0+cu128 | transformers 5.17.0 | device cuda:0 (Tesla T4) | encoders ['modernbert-large', 'nemotron-1b']


## 2 — Data: one question format for routing and choice

Every item becomes `{task, messages, current, question, options {name: description}, answer}`:
- **routing:** the 6,850 training conversations (templates + realistic) with their 5 domain agents (name + description) and the agent currently holding the chat;
- **choice:** the 9,113 general questions; the text is a single user message, there is no current agent, and the options have no description.

**Dev sets** (early stopping only): 10% of the routing **domains** (unseen companies, as in the tests) and 10% of the choice **texts**. **Tests:** E10, E11, banking, E12, exactly as in `modernbert.ipynb` and `decider_benchmark.ipynb`.

In [2]:
DATA_BRANCH = "exp/phase9b-lora-combined"
DATA_DIR = Path("data/modernbert_router")
if not DATA_DIR.exists():
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data" / "modernbert_router"
load = lambda p: json.load(open(DATA_DIR / p))

# ---- routing training data
domains = load("generic/domains.json"); domains.update(load("realistic_train/domains.json"))
route_convs = load("generic/conversations.json") + load("realistic_train/conversations.json")
route_items = [{"task": "routing", "domain": ex["domain"], "messages": ex["messages"], "current": ex["current_agent"],
                "question": ROUTE_Q, "options": domains[ex["domain"]]["agents"], "answer": ex["target_agent"]} for ex in route_convs]
# ---- choice training data
choice_items = [{"task": it["type"], "domain": it["domain"], "messages": [{"role": "user", "content": it["state"]}], "current": None,
                 "question": it["question"], "options": {o: "" for o in it["options"]}, "answer": it["answer"], "state": it["state"]}
                for it in load("choice_train/items.json")["items"]]

rng = random.Random(SEED)
route_domains = sorted({x["domain"] for x in route_items}); rng.shuffle(route_domains)
dev_domains = set(route_domains[: max(1, len(route_domains) // 10)])
texts = sorted({x["state"] for x in choice_items}); rng.shuffle(texts)
dev_texts = set(texts[: len(texts) // 10])
train_route = [x for x in route_items if x["domain"] not in dev_domains]
dev_route = [x for x in route_items if x["domain"] in dev_domains]
train_choice = [x for x in choice_items if x["state"] not in dev_texts]
dev_choice = [x for x in choice_items if x["state"] in dev_texts]

# ---- tests
real_convs = load("realistic/conversations.json")
ft = load("first_turn/messages.json")
banking = load("banking/benchmark.json") + load("banking/extended.json")
e12 = load("choice_test/items.json")["items"]
BANKING_AGENTS = {                                          # exactly as in modernbert.ipynb, Cell 37
    "Payments Agent": "Handles bank transfers, failed or pending payments, transaction status, payment charges and "
                      "duplicate charges, refunds for transactions, and settlement problems where money was sent but "
                      "not received. Does not handle card servicing such as blocking, replacing or activating cards.",
    "Card Support Agent": "Handles debit and credit card servicing: card activation, blocking and unblocking, lost or "
                          "stolen cards, card replacement and delivery, card limits, card PIN, card security and "
                          "compromised card details, and card-specific transactions.",
    "Investment Agent": "Handles mutual funds, SIPs, investments, portfolio allocation, investment transactions, "
                        "redemptions and investment planning.",
    "Insurance Agent": "Handles insurance policies, premiums, renewals, claims, policy expiry and policy servicing.",
    "General Customer Support Agent": "Handles general account questions, profile and contact details, issues that do "
                                      "not clearly belong to a specialist, and routing customers to the right agent.",
}
companies = sorted({c["company"] for c in real_convs})
TEST_COMPANIES = set(companies) - set(companies[::2])           # the same 6 test companies as modernbert.ipynb Cell 70

if SMOKE:
    train_route, dev_route, train_choice, dev_choice = train_route[::400], dev_route[::100], train_choice[::400], dev_choice[::100]
    real_convs, banking, e12 = real_convs[:2], banking[:6], e12[::20]
    ft = {"companies": ft["companies"], "messages": ft["messages"][::20]}

for name, xs in [("train routing", train_route), ("train choice", train_choice), ("dev routing", dev_route), ("dev choice", dev_choice)]:
    print(f"{name:14s}: {len(xs):5d} questions | " + ", ".join(f"{t} {n}" for t, n in Counter(x["task"] for x in xs).items()))
print(f"dev routing domains (unseen in training): {sorted(dev_domains)}")
print(f"tests: E10 {len(real_convs)} conversations / {sum(len(c['user_turns']) for c in real_convs)} turns | E11 {len(ft['messages'])} | "
      f"banking {len(banking)} | E12 {len(e12)}")

Cloning into 'llms-the-hard-way'...


train routing :  6319 questions | routing 6319
train choice  :  8176 questions | noul 4989, choice 2543, score 644
dev routing   :   531 questions | routing 531
dev choice    :   937 questions | noul 583, choice 289, score 65
dev routing domains (unseen in training): ['campsite', 'hearing_clinic', 'home_cinema_installer', 'hr', 'motorbike_rental', 'museum', 'opticians']
tests: E10 35 conversations / 193 turns | E11 100 | banking 225 | E12 170


## 3 — Encoder with LoRA, scoring head, input format

**LoRA** (as in `choice_lora.ipynb`): every chosen linear layer `W` of the frozen encoder gets a trainable update `(alpha / r) · B (A x)`, with `B` starting at zero. ModernBERT: `Wqkv`, `Wo`, `Wi` (attention and MLP, all layers). Nemotron (Llama): `q/k/v/o_proj` and `gate/up/down_proj`. Nemotron also uses **gradient checkpointing**: activations are recomputed in the backward pass, which trades time for memory on the T4.

**Head:** the mean of the output token vectors → `Dropout → Linear(H, 256) → GELU → Dropout → Linear(256, 1)`.

**Input:** `<prefix><role>: <message> SEP ... SEP current agent: X SEP question: Q SEP candidate: <name. description>`; the separator and prefix are the encoder's own (ModernBERT: `[SEP]`; Nemotron: a new line and `query: `).

In [3]:
from transformers import AutoModel, AutoTokenizer

ENCODERS = {
    "modernbert-large": {"name": "answerdotai/ModernBERT-large", "sep": " [SEP] ", "prefix": "", "dtype": torch.float32,
                         "targets": {"Wqkv", "Wo", "Wi"}, "remote": False, "ckpt": False},
    "modernbert-base":  {"name": "answerdotai/ModernBERT-base", "sep": " [SEP] ", "prefix": "", "dtype": torch.float32,
                         "targets": {"Wqkv", "Wo", "Wi"}, "remote": False, "ckpt": False},
    "nemotron-1b":      {"name": "nvidia/llama-nemotron-embed-1b-v2", "sep": "\n", "prefix": "query: ", "dtype": torch.float16,
                         "targets": {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"},
                         "remote": True, "ckpt": True},
}


class LoRALinear(nn.Module):
    """A frozen linear layer plus a trainable low-rank update: y = W x + (alpha / r) * B (A x)."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features, device=base.weight.device))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=base.weight.device))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t().to(x.dtype) @ self.B.t().to(x.dtype)) * self.scale


def add_lora(model, targets):
    found = [(m, n, c) for m in list(model.modules()) for n, c in list(m.named_children()) if n in targets and isinstance(c, nn.Linear)]
    for m, n, c in found:
        setattr(m, n, LoRALinear(c, LORA_R, LORA_ALPHA, LORA_DROPOUT))
    return len(found)


class Model:
    """One encoder with LoRA + one scoring head, and everything needed to score a question."""

    def __init__(self, key):
        self.key, self.spec = key, ENCODERS[key]
        spec = self.spec
        self.tok = AutoTokenizer.from_pretrained(spec["name"], trust_remote_code=spec["remote"])
        self.tok.padding_side, self.tok.truncation_side = "right", "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.enc = AutoModel.from_pretrained(spec["name"], trust_remote_code=spec["remote"], dtype=spec["dtype"]).to(DEVICE)
        self.enc.requires_grad_(False)
        self.n_lora = add_lora(self.enc, spec["targets"])
        assert self.n_lora > 0, f"no LoRA target layers found in {spec['name']}"
        if spec["ckpt"]:
            try:
                self.enc.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            except Exception as e:                           # some remote-code models do not support it
                log(f"gradient checkpointing not available: {e}")
        self.H = self.enc.config.hidden_size
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(self.H, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 1)).to(DEVICE)

    def lora_params(self):
        return [p for p in self.enc.parameters() if p.requires_grad]

    def train(self, mode=True):
        self.enc.train(mode); self.head.train(mode)

    def text(self, item, name):
        sep, desc = self.spec["sep"], item["options"][name]
        parts = [f"{m['role']}: {m['content']}" for m in item["messages"]]
        parts += [f"current agent: {item['current'] or 'none'}", f"question: {item['question']}",
                  f"candidate: {name}. {desc}" if desc else f"candidate: {name}"]
        return self.spec["prefix"] + sep.join(parts)

    def scores(self, items):
        """-> [len(items), K_MAX] scores (padded slots -1e4), one batched encoder pass over every option of every item."""
        inputs, owner, slot = [], [], []
        for b, it in enumerate(items):
            for k, name in enumerate(it["options"]):
                inputs.append(self.text(it, name)); owner.append(b); slot.append(k)
        enc = self.tok(inputs, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            hidden = self.enc(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).float()
        s = self.head((hidden.float() * m).sum(1) / m.sum(1)).squeeze(-1)
        out = torch.full((len(items), K_MAX), -1e4, device=DEVICE)
        out[torch.tensor(owner, device=DEVICE), torch.tensor(slot, device=DEVICE)] = s
        return out

    def probs(self, item):
        """Probabilities over one item's options (in order), no gradient."""
        with torch.no_grad():
            return torch.softmax(self.scores([item]), dim=1)[0, :len(item["options"])].float().cpu()

## 4 — Training and evaluation functions

**Training.** Each epoch shuffles routing batches (`BATCH_ROUTE` questions) and choice batches (`BATCH_CHOICE` questions) together. The loss is listwise cross-entropy over each question's options. AdamW (separate learning rates for adapters and head), 6% warm-up then linear decay, gradient clipping 1.0, fp16 mixed precision. After each epoch both dev sets are scored. The adapters and head with the best **mean** of routing-dev and choice-dev accuracy are kept.

**Evaluation.** Each test question is scored on its own (batch = one question, all its options in one pass), and timed with the GPU synchronized, tokenization included:
- **E10:** live. The model's own previous pick is the current agent, so mistakes can carry over. Reported per turn type, for all 193 turns and for the 83 test-company turns.
- **E11:** first message only, no current agent; top-1 and top-2.
- **Banking:** the last user message of each conversation, with its own current agent.
- **E12:** yes / no, choice, score, per category, the same-text question groups, and the mistakes.

In [4]:
def lat_summary(ms):
    ms = sorted(ms)
    return {"p50": ms[len(ms) // 2], "p95": ms[min(len(ms) - 1, int(0.95 * len(ms)))], "mean": sum(ms) / len(ms), "n": len(ms)}


def timed_probs(model, item):
    if AMP:
        torch.cuda.synchronize(DEVICE)
    t0 = time.perf_counter()
    p = model.probs(item)
    if AMP:
        torch.cuda.synchronize(DEVICE)
    return p, (time.perf_counter() - t0) * 1000


def dev_accuracy(model, items, batch=16):
    model.train(False)
    ok = 0
    with torch.no_grad():
        for s0 in range(0, len(items), batch):
            chunk = items[s0:s0 + batch]
            pick = model.scores(chunk).argmax(1).tolist()
            ok += sum(list(it["options"])[k] == it["answer"] for it, k in zip(chunk, pick))
    return ok / max(1, len(items)), ok


def train_model(model):
    batches = [train_route[i:i + BATCH_ROUTE] for i in range(0, len(train_route), BATCH_ROUTE)]
    batches += [train_choice[i:i + BATCH_CHOICE] for i in range(0, len(train_choice), BATCH_CHOICE)]
    params = model.lora_params() + list(model.head.parameters())
    opt = torch.optim.AdamW([{"params": model.lora_params(), "lr": LR}, {"params": model.head.parameters(), "lr": HEAD_LR}], weight_decay=0.01)
    total = EPOCHS * len(batches)
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best, step, t0 = (-1.0, None, 0), 0, time.perf_counter()
    log(f"[{model.key}] training: {len(batches)} batches per epoch ({len(train_route)} routing + {len(train_choice)} choice questions), "
        f"{total} steps")
    for epoch in range(1, EPOCHS + 1):
        model.train(True)
        order = list(range(len(batches)))
        random.Random(SEED + epoch).shuffle(order)
        run = {"routing": [], "choice": []}
        for bi in order:
            batch = batches[bi]
            items = [dict(it) for it in batch]
            for it in items:                                    # option order shuffled, so position can't be learned
                names = list(it["options"]); random.shuffle(names); it["options"] = {n: it["options"][n] for n in names}
            target = torch.tensor([list(it["options"]).index(it["answer"]) for it in items], device=DEVICE)
            loss = F.cross_entropy(model.scores(items), target)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run["routing" if batch[0]["task"] == "routing" else "choice"].append(loss.item())
            step += 1
            if step % 100 == 0 or step == total:
                el = time.perf_counter() - t0
                log(f"  [{model.key}] epoch {epoch} step {step}/{total} | loss routing {statistics.mean(run['routing'][-50:] or [0]):.4f} "
                    f"choice {statistics.mean(run['choice'][-50:] or [0]):.4f} | {step / el:.2f} steps/s | "
                    f"~{(total - step) / (step / el) / 60:.1f} min left | {gpu()}")
        acc_r, ok_r = dev_accuracy(model, dev_route)
        acc_c, ok_c = dev_accuracy(model, dev_choice)
        log(f"[{model.key}] epoch {epoch} done | dev routing {ok_r}/{len(dev_route)} ({acc_r:.1%}) | dev choice {ok_c}/{len(dev_choice)} ({acc_c:.1%})")
        if (acc_r + acc_c) / 2 > best[0]:
            best = ((acc_r + acc_c) / 2, {"lora": {n: p.detach().clone() for n, p in model.enc.named_parameters() if p.requires_grad},
                                          "head": copy.deepcopy(model.head.state_dict())}, epoch)
    params_by_name = dict(model.enc.named_parameters())
    for n, v in best[1]["lora"].items():
        params_by_name[n].data.copy_(v)
    model.head.load_state_dict(best[1]["head"])
    minutes = (time.perf_counter() - t0) / 60
    log(f"[{model.key}] training done in {minutes:.1f} min | best epoch {best[2]} (mean dev accuracy {best[0]:.1%})")
    return {"train_minutes": minutes, "best_epoch": best[2]}


TURN_TYPES = ["start", "answer", "follow", "aspect", "closing", "also_stay", "switch", "short_switch", "return", "handoff_ok", "branch"]


def eval_e10(model):
    model.train(False)
    rows, lat = [], []
    for conv in real_convs:
        cur = None
        for u in conv["user_turns"]:
            prefix = conv["messages"][:u["message_index"] + 1]
            item = {"messages": prefix, "current": cur, "question": ROUTE_Q, "options": conv["agents"]}
            p, ms = timed_probs(model, item)
            lat.append(ms)
            cur = list(conv["agents"])[int(p.argmax())]
            rows.append({"company": conv["company"], "type": u["turn_type"], "ok": cur == u["expected_agent"]})
    test = [r for r in rows if r["company"] in TEST_COMPANIES]
    print(f"E10 live routing | all {sum(r['ok'] for r in rows)}/{len(rows)} | test companies {sum(r['ok'] for r in test)}/{len(test)}")
    print(f"  {'turn type':14s} | {'all':>9} | {'test':>9}")
    for tt in TURN_TYPES:
        a, t_ = [r for r in rows if r["type"] == tt], [r for r in test if r["type"] == tt]
        if a:
            print(f"  {tt:14s} | {sum(r['ok'] for r in a):>3} / {len(a):<4} | {sum(r['ok'] for r in t_):>3} / {len(t_):<4}")
    return {"all": sum(r["ok"] for r in rows), "all_n": len(rows), "test": sum(r["ok"] for r in test), "test_n": len(test),
            "latency": lat_summary(lat)}


def eval_e11(model):
    model.train(False)
    top1 = top2 = 0
    lat = []
    for m in ft["messages"]:
        agents = ft["companies"][m["company"]]
        item = {"messages": [{"role": "user", "content": m["message"]}], "current": None, "question": ROUTE_Q, "options": agents}
        p, ms = timed_probs(model, item)
        lat.append(ms)
        names = list(agents)
        top1 += names[int(p.argmax())] == m["target_agent"]
        top2 += m["target_agent"] in [names[i] for i in p.topk(2).indices.tolist()]
    print(f"E11 first turn | top-1 {top1}/{len(ft['messages'])} | top-2 {top2}/{len(ft['messages'])}")
    return {"top1": top1, "top2": top2, "n": len(ft["messages"]), "latency": lat_summary(lat)}


def eval_banking(model):
    model.train(False)
    ok, lat = [], []
    for ex in banking:
        item = {"messages": ex["messages"], "current": ex["current_agent"], "question": ROUTE_Q, "options": BANKING_AGENTS}
        p, ms = timed_probs(model, item)
        lat.append(ms)
        ok.append(list(BANKING_AGENTS)[int(p.argmax())] == ex["target_agent"])
    bench = sum(ok[:50])
    print(f"banking | benchmark {bench}/{min(50, len(ok))} | all {sum(ok)}/{len(ok)}")
    return {"benchmark": bench, "all": sum(ok), "n": len(ok), "latency": lat_summary(lat)}


def eval_e12(model):
    model.train(False)
    rows, lat = [], []
    for it in e12:
        item = {"messages": [{"role": "user", "content": it["state"]}], "current": None, "question": it["question"],
                "options": {o: "" for o in it["options"]}}
        p, ms = timed_probs(model, item)
        lat.append(ms)
        top = int(p.argmax())
        row = {**it, "pick": it["options"][top], "ok": it["options"][top] == it["answer"], "probs": p.tolist()}
        if it["type"] == "score":
            row["level_error"] = abs(top - it["answer_index"])
            row["expected_level"] = float(sum(i * pi for i, pi in enumerate(p.tolist())))
        rows.append(row)
    by = {t: sum(r["ok"] for r in rows if r["type"] == t) for t in ["noul", "choice", "score"]}
    sc = [r for r in rows if r["type"] == "score"]
    groups = {}
    for r in rows:
        if "pair" in r:
            groups.setdefault(r["pair"], []).append(r)
    g_ok = sum(all(r["ok"] for r in g) for g in groups.values())
    print(f"E12 choice | all {sum(r['ok'] for r in rows)}/{len(rows)} | yes/no {by['noul']} | choice {by['choice']} | score {by['score']} "
          f"(within 1 level {sum(r['level_error'] <= 1 for r in sc)}/{len(sc)}) | question groups {g_ok}/{len(groups)} | "
          f"answered yes {sum(r['pick'] == 'yes' for r in rows if r['type'] == 'noul')} (truth {sum(r['answer'] == 'yes' for r in rows if r['type'] == 'noul')})")
    print("  per category: " + ", ".join(f"{c} {sum(r['ok'] for r in rows if r['category'] == c)}/{sum(r['category'] == c for r in rows)}"
                                         for c in sorted({r['category'] for r in rows})))
    print("  mistakes:")
    for r in [r for r in rows if not r["ok"]]:
        s_ = r["state"] if len(r["state"]) <= 44 else r["state"][:41] + "..."
        print(f"    {r['id']} {r['type']:6s} | {s_:44s} | {r['question'][:36]:36s} | expected {r['answer']:14s} | got {r['pick']} ({max(r['probs']):.2f})")
    return {"all": sum(r["ok"] for r in rows), "n": len(rows), **by, "groups": g_ok, "latency": lat_summary(lat)}


def evaluate_all(model):
    model.train(False)
    model.probs({"messages": [{"role": "user", "content": "warm-up"}], "current": None, "question": ROUTE_Q, "options": BANKING_AGENTS})
    res = {}
    for name, fn in [("E10", eval_e10), ("E11", eval_e11), ("banking", eval_banking), ("E12", eval_e12)]:
        log(f"[{model.key}] evaluating {name}")
        res[name] = fn(model)
        L = res[name]["latency"]
        print(f"  latency {name}: p50 {L['p50']:.0f} ms | p95 {L['p95']:.0f} ms | mean {L['mean']:.0f} ms ({L['n']} questions)\n")
    return res


def run_encoder(key):
    """Build, train and evaluate one encoder; free the GPU afterwards. -> results dict (also saved to JSON)."""
    log(f"===== {key} =====")
    if AMP:
        torch.cuda.reset_peak_memory_stats(DEVICE)
    model = Model(key)
    log(f"[{key}] {model.spec['name']}: {model.n_lora} LoRA layers | trainable {sum(p.numel() for p in model.lora_params()) / 1e6:.1f} M "
        f"+ head {sum(p.numel() for p in model.head.parameters()) / 1e6:.2f} M | {gpu()}")
    res = {"encoder": key, **train_model(model)}
    res.update(evaluate_all(model))
    res["peak_gpu_gb"] = torch.cuda.max_memory_allocated(DEVICE) / 1e9 if AMP else 0
    with open(f"lora_combined_{key}.json", "w") as fh:
        json.dump(res, fh, indent=1)
    del model
    gc.collect()
    if AMP:
        torch.cuda.empty_cache()
    return res


RESULTS = {}

## 5 — ModernBERT-large: train and evaluate

**Expect.** Progress every 100 steps, dev accuracy per epoch, then the four tests with their latencies. Results are saved to `lora_combined_modernbert-large.json`.

In [5]:
if RUN_ENCODERS[0] in ENCODERS:
    RESULTS[RUN_ENCODERS[0]] = run_encoder(RUN_ENCODERS[0])

[15:52:38] ===== modernbert-large =====


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.58GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[15:52:53] [modernbert-large] answerdotai/ModernBERT-large: 112 LoRA layers | trainable 7.2 M + head 0.26 M | GPU 1.6 GB (peak 1.6)
[15:52:53] [modernbert-large] training: 2091 batches per epoch (6319 routing + 8176 choice questions), 4182 steps
[15:53:33]   [modernbert-large] epoch 1 step 100/4182 | loss routing 1.4704 choice 0.8354 | 2.51 steps/s | ~27.1 min left | GPU 1.7 GB (peak 8.5)
[15:54:14]   [modernbert-large] epoch 1 step 200/4182 | loss routing 0.9945 choice 0.7537 | 2.47 steps/s | ~26.9 min left | GPU 1.7 GB (peak 9.0)
[15:55:01]   [modernbert-large] epoch 1 step 300/4182 | loss routing 0.8423 choice 0.6373 | 2.34 steps/s | ~27.6 min left | GPU 1.7 GB (peak 9.6)
[15:55:45]   [modernbert-large] epoch 1 step 400/4182 | loss routing 0.5519 choice 0.6840 | 2.33 steps/s | ~27.1 min left | GPU 1.7 GB (peak 9.6)
[15:56:30]   [modernbert-large] epoch 1 step 500/4182 | loss routing 1.2338 choice 0.7034 | 2.30 steps/s | ~26.6 min left | GPU 1.7 GB (peak 9.6)
[15:57:14]   [modernbert

## 6 — Nemotron-1B: train and evaluate

The same data, settings and tests. Nemotron is about three times larger, so training is slower (gradient checkpointing on). If it runs out of GPU memory, rerun this cell with a smaller `BATCH_ROUTE` (e.g. `os.environ["BATCH_ROUTE"] = "2"` and re-execute Section 1), and the ModernBERT results above are kept.

In [6]:
for key in RUN_ENCODERS[1:]:
    RESULTS[key] = run_encoder(key)

[16:25:11] ===== nemotron-1b =====


config.json:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

llama_bidirectional_model.py:   0%|          | 0.00/9.06k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/nvidia/llama-nemotron-embed-1b-v2:
- llama_bidirectional_model.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/449 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[16:25:32] [nemotron-1b] nvidia/llama-nemotron-embed-1b-v2: 112 LoRA layers | trainable 11.3 M + head 0.52 M | GPU 2.5 GB (peak 2.5)
[16:25:32] [nemotron-1b] training: 2091 batches per epoch (6319 routing + 8176 choice questions), 4182 steps
[16:27:09]   [nemotron-1b] epoch 1 step 100/4182 | loss routing 1.4658 choice 0.8287 | 1.03 steps/s | ~66.1 min left | GPU 2.7 GB (peak 3.5)
[16:28:47]   [nemotron-1b] epoch 1 step 200/4182 | loss routing 0.8897 choice 0.7875 | 1.02 steps/s | ~64.8 min left | GPU 2.7 GB (peak 3.5)
[16:30:27]   [nemotron-1b] epoch 1 step 300/4182 | loss routing 0.9466 choice 0.7626 | 1.02 steps/s | ~63.6 min left | GPU 2.7 GB (peak 3.6)
[16:32:03]   [nemotron-1b] epoch 1 step 400/4182 | loss routing 0.4019 choice 0.7562 | 1.02 steps/s | ~61.6 min left | GPU 2.7 GB (peak 3.6)
[16:33:40]   [nemotron-1b] epoch 1 step 500/4182 | loss routing 0.4842 choice 0.8543 | 1.03 steps/s | ~59.8 min left | GPU 2.7 GB (peak 3.6)
[16:35:15]   [nemotron-1b] epoch 1 step 600/4182 | lo

## 7 — Side by side: both fine-tuned models, the frozen system and the decider

The reference numbers come from the experiment ledger: run 17 (frozen Nemotron + trained heads; E10 router alone), run 18 (choice-only LoRA), and the Strands decider (B1, plus `decider_benchmark.ipynb` section 10). The decider's latency on routing comes from its section 11 (p50 / p95).

In [7]:
REF = {
    "frozen system (run 17)": {"E10 test (83)": 69, "E10 all (193)": 158, "E11 top-1 (100)": 94, "banking 50": 43, "banking 225": 194, "E12 (170)": 114},
    "choice-only LoRA (run 18)": {"E12 (170)": 153},
    "Strands decider 2B": {"E10 test (83)": "62 / 69 (w3)", "E10 all (193)": "151 / 163 (w3)", "E11 top-1 (100)": 96, "banking 50": 48,
                           "banking 225": 214, "E12 (170)": 160},
}
rows = ["E10 test (83)", "E10 all (193)", "E11 top-1 (100)", "banking 50", "banking 225", "E12 (170)"]


def ours(r, row):
    return {"E10 test (83)": r["E10"]["test"], "E10 all (193)": r["E10"]["all"], "E11 top-1 (100)": r["E11"]["top1"],
            "banking 50": r["banking"]["benchmark"], "banking 225": r["banking"]["all"], "E12 (170)": r["E12"]["all"]}[row]


cols = [f"LoRA {k}" for k in RESULTS] + list(REF)
print(f"{'accuracy':18s} | " + " | ".join(f"{c:>26s}" for c in cols))
for row in rows:
    vals = [ours(RESULTS[k], row) for k in RESULTS] + [REF[c].get(row, "—") for c in REF]
    print(f"{row:18s} | " + " | ".join(f"{str(v):>26s}" for v in vals))

print(f"\n{'latency (ms)':18s} | " + " | ".join(f"{('LoRA ' + k):>26s}" for k in RESULTS))
for test in ["E10", "E11", "banking", "E12"]:
    print(f"{test + ' p50 / p95':18s} | " + " | ".join(f"{RESULTS[k][test]['latency']['p50']:>15.0f} / {RESULTS[k][test]['latency']['p95']:<8.0f}" for k in RESULTS))
print(f"{'training (min)':18s} | " + " | ".join(f"{RESULTS[k]['train_minutes']:>26.1f}" for k in RESULTS))
print(f"{'peak GPU (GB)':18s} | " + " | ".join(f"{RESULTS[k]['peak_gpu_gb']:>26.1f}" for k in RESULTS))

accuracy           |      LoRA modernbert-large |           LoRA nemotron-1b |     frozen system (run 17) |  choice-only LoRA (run 18) |         Strands decider 2B
E10 test (83)      |                         75 |                         79 |                         69 |                          — |               62 / 69 (w3)
E10 all (193)      |                        174 |                        180 |                        158 |                          — |             151 / 163 (w3)
E11 top-1 (100)    |                         96 |                         97 |                         94 |                          — |                         96
banking 50         |                         49 |                         48 |                         43 |                          — |                         48
banking 225        |                        212 |                        217 |                        194 |                          — |                        214
E12 (170)       